# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [1]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [2]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant
# termianal에서 실행

## 0. 패키지 설치

In [3]:
# 필요한 library: pymupdf pdfplumber pillow openai qdrant-client langchain-core langchain-openai langchain-qdrant langchain-text-splitters python-dotenv


## 1. 설정 및 클라이언트 준비


In [4]:
import os
import re
import glob
import uuid
from pathlib import Path

import fitz  # PyMuPDF
import pdfplumber
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# ----- 설정값 -----
GUIDANCE_DIR = "./길라잡이"          # 이 폴더 안의 모든 PDF를 찾아서 적재

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = "guidance_vectordb"

CHAT_MODEL = "gpt-5.4-mini"                          # 표 요약 / 이미지 캐법션용
EMBED_MODEL = "text-embedding-3-large"         # large 임베딩 모델
EMBED_DIM = 3072

CHUNK_SIZE = 800       # 텍스트 청크 최대 글자 수 (splitter 기준)
CHUNK_OVERLAP = 150    # 청크 간 격치는 글자 수 (문맥 끓김 방지)

openai_client = OpenAI()
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def ensure_collection() -> bool:
    """collection이 이미 있으면 True(기존 데이터 존재), 없으면 새로 만들고 False를 반환"""
    existing = [c.name for c in qdrant_client.get_collections().collections]
    if COLLECTION_NAME in existing:
        print(f"'{COLLECTION_NAME}' collection이 이미 존재합니다.")
        return True
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )
    print(f"'{COLLECTION_NAME}' collection 생성 완료 (dim={EMBED_DIM})")
    return False


collection_already_existed = ensure_collection()

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

pdf_paths = sorted(glob.glob(str(Path(GUIDANCE_DIR) / "*.pdf")))
print(f"'{GUIDANCE_DIR}' 안에서 발견된 PDF {len(pdf_paths)}개")
for p in pdf_paths:
    print(" -", p)


'guidance_vectordb' collection이 이미 존재합니다.
'./길라잡이' 안에서 발견된 PDF 0개


## 2. 텍스트 정제

In [5]:
def clean_text(text: str) -> str:
    # Private Use Area (U+E000–U+F8FF) 문자 제거
    text = re.sub(r'[\ue000-\uf8ff]', '', text)
    # 과도한 공백/빈 줄 정리
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


## 3. 표 추출 → 마크다운 → 요약

In [6]:
def table_to_markdown(table_rows):
    rows = [[cell if cell is not None else "" for cell in row] for row in table_rows]
    if not rows:
        return ""
    header = rows[0]
    body = rows[1:]
    md_lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for row in body:
        md_lines.append("| " + " | ".join(row) + " |")
    return "\n".join(md_lines)


def extract_tables(pdf_path: str):
    tables_out = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                md_table = table_to_markdown(table)
                tables_out.append({"page": page_num, "markdown": md_table})
    return tables_out


def summarize_table(table_markdown: str) -> str:
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        max_completion_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                "다음은 문서에서 추출한 표입니다. 이 표가 어떤 내용을 담고 있는지 "
                "검색에 활용할 수 있도록 2~4문장의 한국어로 요약해줘. "
                "컬럼이 의미하는 바와 주요 수치/항목을 포함해줘.\n\n"
                f"{table_markdown}"
            ),
        }],
    )
    return response.choices[0].message.content.strip()


## 4. 텍스트 청킹 (페이지 단위)

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# 이 길이를 넘는 페이지만 splitter로 추가 분할 (그 외엔 페이지 통째로 한 청크)
PAGE_OVERFLOW_THRESHOLD = 1200


def chunk_text_by_page(pdf_path: str) -> list[dict]:
    """페이지 단위로 청크를 만든다. 페이지가 너무 길 때만 그 페이지 안에서 추가로 나눈다."""
    doc = fitz.open(pdf_path)
    results = []

    for page_num, page in enumerate(doc, start=1):
        page_text = clean_text(page.get_text("text"))

        if not page_text:
            continue  # 텍스트 없는 페이지(이미지만 있는 페이지 등)는 건너뜀

        if len(page_text) <= PAGE_OVERFLOW_THRESHOLD:
            results.append({"page": page_num, "text": page_text})
        else:
            # 이 페이지만 너무 길어서 임베딩이 흐려질 수 있는 경우 -> 페이지 내부에서만 분할
            sub_chunks = text_splitter.split_text(page_text)
            for sub in sub_chunks:
                results.append({"page": page_num, "text": sub})

    doc.close()
    return results


## 5. PDF 1개 → LangChain `Document` 리스트로 통합

In [8]:
ID_NAMESPACE = uuid.NAMESPACE_URL


def make_id(doc_name: str, chunk_type: str, index: int) -> str:
    """같은 PDF를 다시 적재해도 같은 ID가 나오도록 결정론적으로 생성 (재실행 시 upsert로 덮어씀)"""
    return str(uuid.uuid5(ID_NAMESPACE, f"{doc_name}::{chunk_type}::{index}"))


def build_documents_for_pdf(pdf_path: str) -> tuple[list[Document], list[str]]:
    doc_name = Path(pdf_path).name
    documents = []
    ids = []
    counter = 0

    # 1) 텍스트 청크
    text_chunks = chunk_text_by_page(pdf_path)
    for c in text_chunks:
        documents.append(Document(
            page_content=c["text"],
            metadata={"doc_name": doc_name, "page": c["page"], "type": "text"},
        ))
        ids.append(make_id(doc_name, "text", counter))
        counter += 1

    # 2) 표
    tables = extract_tables(pdf_path)
    for t in tables:
        summary = summarize_table(t["markdown"])
        documents.append(Document(
            page_content=summary,
            metadata={
                "doc_name": doc_name, "page": t["page"], "type": "table",
                "table_markdown": t["markdown"], "summary": summary,
            },
        ))
        ids.append(make_id(doc_name, "table", counter))
        counter += 1

    print(f"[{doc_name}] 텍스트 {len(text_chunks)} / 표 {len(tables)} → 총 {len(documents)}개")
    return documents, ids


## 6. 디렉토리 전체 적재 (upsert)

In [9]:
def upsert_documents(documents: list[Document], ids: list[str], batch_size: int = 100):
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
        print(f"  upsert 진행: {min(i + batch_size, len(documents))}/{len(documents)}")


def run_pipeline(directory: str = GUIDANCE_DIR):
    paths = sorted(glob.glob(str(Path(directory) / "*.pdf")))
    print(f"'{directory}'에서 PDF {len(paths)}개 발견, 적재 시작\n")

    total_chunks = 0
    for pdf_path in paths:
        documents, ids = build_documents_for_pdf(pdf_path)
        upsert_documents(documents, ids)
        total_chunks += len(documents)
        print()

    print(f"완료: PDF {len(paths)}개 / 총 {total_chunks}개 청크 적재")


# collection이 이미 있었다면 재적재를 건너뜀 (PDF 파싱/표 요약/임베딩 API 재호출 비용 방지)
if not collection_already_existed:
    run_pipeline(GUIDANCE_DIR)
else:
    print(f"'{COLLECTION_NAME}' 컴렉션에 이미 데이터가 있어 적재를 건너뜁니다. (재적재하려면 run_pipeline(GUIDANCE_DIR)를 직접 호출하세요)")


'guidance_vectordb' 컴렉션에 이미 데이터가 있어 적재를 건너뜁니다. (재적재하려면 run_pipeline(GUIDANCE_DIR)를 직접 호출하세요)


## 7. 검색 예시

In [10]:
def search(query: str, k: int = 5, type_filter: str = None):
    qfilter = None
    if type_filter:
        qfilter = Filter(must=[FieldCondition(key="metadata.type", match=MatchValue(value=type_filter))])

    results = vectorstore.similarity_search_with_score(query, k=k, filter=qfilter)

    for doc, score in results:
        print(f"[score={score:.3f}] doc={doc.metadata.get('doc_name')} page={doc.metadata.get('page')} type={doc.metadata.get('type')}")
        print(doc.page_content[:200])
        print("-" * 40)

    return results


# 사용 예시:
search("군인 해택 휴양시설알려줘")
# search("휴가 관련 표", type_filter="table")
# search("기차 할인 안내 이미지", type_filter="image")



[score=0.585] doc=사병_길라잡이.pdf page=23 type=text
2026 병 복지 길라잡이 23
 • 국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
822-6470
033-631-9331
송 정 콘도
강원 강릉시 
940-38
----------------------------------------
[score=0.549] doc=사병_길라잡이.pdf page=22 type=text
Part 01 생활편의 분야
22 2026 병 복지 길라잡이
휴양시설 이용
이용대상 / 방법 및 혜택
 • 이용대상 : 현역병, 배우자 및 그 직계 존·비속
 * 가족의 경우 가족관계 증빙서류, 신분증 또는 국방가족 모바일증명 제시
 * 국방가족 모바일증명은 국군복지포털 홈페이지를 이용하여 등록
 • 국군복지단 휴양시설 이용방법
 - 국군복지포털 홈페이지 
----------------------------------------
[score=0.543] doc=초급간부_길라잡이.pdf page=35 type=text
34 
2026 초급간부 길라잡이
•국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
988-7873
033-631-9331
송정 콘도
강원 강릉시 
940-3881
----------------------------------------
[score=0.525] doc=초급간부_길라잡이.pdf page=34 type=text
02
Part
생
활
편
의
 
분
야
33 
2026 초급간부 길라잡이
•국군복지단 초급간부전용 객실예약 이용방법
- 국군복지포털 호텔/

[(Document(metadata={'doc_name': '사병_길라잡이.pdf', 'page': 23, 'type': 'text', '_id': 'fb4aa98d-c931-5260-82b6-e1fa8da946b5', '_collection_name': 'guidance_vectordb'}, page_content='2026 병 복지 길라잡이 23\n • 국군복지단 휴양시설 현황\n구분\n시 설 명\n위 치\n전화번호(군)\n전화번호(일반)\n직영\n서귀포 호텔\n제주 서귀포시\n960-7703\n064-738-0123\n화진포 콘도\n강원 고성군 \n960-7706\n033-682-0500\n청간정 콘도\n강원 고성군 \n822-6470\n033-631-9331\n송 정 콘도\n강원 강릉시 \n940-3881\n033-652-7573\n대 천 콘도\n충남 보령시 \n960-7705\n041-932-6305\n민영\n한 화\n설악, 용인, 대천, 산정호수, \n해운대, 경주, 제주, 평창, 거제\n국군복지단\n예약실\n984-6584\n국군복지단\n예약실\n1577-9800\n내선 “1번”\n금 호\n통영, 화순, 설악, 제주, 아산\n소 노\n설악, 경주, 단양, 홍천, 양양, \n변산, 거제, 삼척, 진도, 여수, \n고양(호텔), 양평\n켄싱턴\n남원, 설악비치, 충주, 경주,\n가평, 설악밸리, 하동, 서귀포\n무 주\n전북무주\n리 솜\n안면도\n엘도라도\n전남신안\n샤인빌(소노캄)\n제주\n보 광\n제주\n웰리힐리\n강원횡성\n영랑호\n강원 속초\n현대수\n강원 속초\n • 각 군 휴양시설 이용방법 : 각 군 홈페이지 참조\n구분\n시 설 명\n홈페이지 주소\n국방부\n국방컨벤션(웨딩)\nwww.mndconvention.co.kr\n육군\n로카우스 호텔, 계룡스파텔\nwelfare.army.mil.kr\n해군\n해군호텔(서울, 평택, 제주), \n해군회관(진해)\nwelfare.navy.mil.kr\n공군\n공군호텔\nwww.airforcehote

## 8. Retriever 생성

In [11]:
##################################################
# retriever 생성
##################################################

def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever


retriever = get_retriever(vectorstore)
print(retriever)


tags=['QdrantVectorStore', 'OpenAIEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000001E5BA7FB230> search_kwargs={'k': 5}


In [12]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(template=prompt_txt)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()


def format_docs_for_prompt(documents: list[Document]) -> str:
    """LLM 프롬프트({context})용: 문서들을 하나의 문자열로 합침 (doc_name/page도 같이 표기)"""
    parts = []
    for doc in documents:
        meta = doc.metadata
        header = f"[{meta.get('doc_name', '')} p.{meta.get('page', '')}]"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def format_docs_for_ragas(documents: list[Document]) -> list[str]:
    """RAGAS 평가용: 검색된 문서 각각의 내용을 list[str]로 추출"""
    return [doc.page_content for doc in documents]


def build_prompt_input(state: dict) -> dict:
    """검색 결과({'documents':..., 'query':...})를 prompt 입력({'context':str, 'query':str})으로 변환"""
    return {
        "context": format_docs_for_prompt(state["documents"]),
        "query": state["query"],
    }


# 1단계: 질문으로 검색 → 원본 Document 리스트 + 질문 보존 (retriever는 여기서 1회만 호출됨)
retrieve_step = {
    "documents": retriever,
    "query": RunnablePassthrough(),
}

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM 응답(str), 검색한 문서들(list[str]), 질문(str)
chain = (
    RunnablePassthrough()  # dict | dict 가 파이썬 dict 병합 연산으로 취급되는 것을 막기 위해 필요
    | retrieve_step
    | {
        "response": RunnableLambda(build_prompt_input) | prompt | model | parser,
        "retrieved_context": RunnableLambda(lambda state: format_docs_for_ragas(state["documents"])),
        "query": RunnableLambda(lambda state: state["query"]),
    }
)


In [13]:
res = chain.invoke("병장의 월급은?")

In [14]:
res["response"]

'병장의 월급은 **1,500,000원**입니다.'

In [15]:
res["retrieved_context"]

['Part 01 생활편의 분야\n6 2026 병 복지 길라잡이\n보 수\n봉 급\n • 계급별 지급액\n 단위 : 원\n \n이병\n일병\n상병\n병장\n750,000\n900,000\n1,200,000\n1,500,000\n • 병 봉급 인상\n 단위 : 원\n연도\n계급\n’21년\n’22년\n’23년\n’24년\n’25~’26년\n병 장 \n608,500\n676,100\n1,000,000\n1,250,000\n1,500,000\n상 병 \n549,200\n610,200\n800,000\n1,000,000\n1,200,000\n일 병\n496,900\n552,100\n680,000\n800,000\n900,000\n이 병\n459,100\n510,100\n600,000\n640,000\n750,000\n인상율(병장)\n12.5%\n11.1%\n47.9%\n25%\n20%\n* 병역의무 이행자의 보상과 예우를 위해 병 봉급과 자산형성프로그램을 결합하여\n’25년까지 병장 기준 200만원 이상으로 인상\n수 당\n단위 : 원\n \n구 분\n지급 기준\n금액\n특수지\n근 무\n수 당\n갑\n・비무장지대, 서해5도, 울릉도, 접적 해역 상주 근무자\n * 가산금 : 서해5도 40,000원, 비무장지대 및 \n북방한계선 인접해역 20,000원\n25,000 \n(가산금별도)\n을\n・GOP, 해안초소, 해발 800m 이상 고지대 상주 근무자\n * 가산금 : 10,000원(GOP / 해안초소)\n20,000\n(가산금별도)\n항 공\n수 당\n을\n・항공기 동승근무자\n40,000 \n함 정\n수 당\n을\n・전투함 및 지원함 승무원\n * 함정출동가산금 : 1일 4,000원\n32,700 \n(가산금별도)\n병\n・수륙양용궤도차량 승무원\n32,700 \npart 01\n생활편의 분야\n▶▶▶ 병 복지혜택 이렇게 다양합니다',
 '이 표는 **계급별 연도(’21~’26년) 급여 수준과 병장 급여 인상률**을 비교한 자료입니다. 행은 **병장·상병·일병·이병

# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>          from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [16]:
# !uv pip install langchain_google_vertexai

In [17]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [18]:
from ragas.testset import TestsetGenerator

c:\Users\Playdata\SKN\SKN31-3rd-2Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답 답변 생성

import random

# 데이터셋을 생성할 때 사용할 Context를 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "guidance_vectordb"

info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs,
)

# random하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" =random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]



In [20]:
total_docs

337

In [21]:
sample_docs

[Record(id='c49f6187-527e-5583-b3ba-f909b7dbb552', payload={'page_content': '생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 지원제도\n-⑥ 맞춤형복지제도\n-⑦ 초급간부 재정안정지원\n-⑧ 군인공제회 회원복지부조\n-⑨ 군인공제회 건강검진 제휴 서비스\n-⑩ 군인공제회 콘도 및 제휴 복지 서비스\n-⑪ 장병e음 플랫폼 서비스\n02\nPart', 'metadata': {'doc_name': '초급간부_길라잡이.pdf', 'page': 26, 'type': 'text'}}, vector=None, shard_key=None, order_value=None),
 Record(id='2e74ecdb-3026-517a-983a-8f527cb7765f', payload={'page_content': '이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 보여줍니다. 표에 나타난 내용은 **채용 설명회**이며, 개막식과 연계된 행사로 채용 관련 정보를 제공하는 프로그램이 있음을 의미합니다. 별도의 수치나 추가 항목은 없고, 핵심 항목은 **개막식 → 채용 설명회**입니다.', 'metadata': {'doc_name': '초급간부_길라잡이.pdf', 'page': 65, 'type': 'table', 'table_markdown': '| 개막식 |\n| --- |\n| 채용\n설명회 |', 'summary': '이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 보여줍니다. 표에 나타난 내용은 **채용 설명회**이며, 개막식과 연계된 행사로 채용 관련 정보를 제공하는 프로그램이 있음을 의미합니다. 별도의 수치나 추가 항목은 없고, 핵심 항목은 **개막식 → 채용 설명회**입니다.'}}, vector=None, shard_key=None, orde

In [22]:
docs

['생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 지원제도\n-⑥ 맞춤형복지제도\n-⑦ 초급간부 재정안정지원\n-⑧ 군인공제회 회원복지부조\n-⑨ 군인공제회 건강검진 제휴 서비스\n-⑩ 군인공제회 콘도 및 제휴 복지 서비스\n-⑪ 장병e음 플랫폼 서비스\n02\nPart',
 '이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 보여줍니다. 표에 나타난 내용은 **채용 설명회**이며, 개막식과 연계된 행사로 채용 관련 정보를 제공하는 프로그램이 있음을 의미합니다. 별도의 수치나 추가 항목은 없고, 핵심 항목은 **개막식 → 채용 설명회**입니다.',
 '01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n/\n2026 초급간부 길라잡이\n01\nPart\n 개 요 \n\x07병과별 균형발전 및 안정적 인력운영을 위한 제도로, 임관 이후에도 \n실무경험과 적성 등을 고려하여 병과 변경 가능\n 주요내용\n•\x07지원자격 \n구 분\n장 교\n부사관\n육 군\n원병과에서 1년 이상 근무\n희망병과 관련 \n자격 취득 및\n교육 이수자 등\n해 군\n임관 1년 ~ 소령 진급심사 1년 전\n공 군\n임관 이후\n해 병\n임관 1년 ~ 소령 진급심사 1년 전\n•\x07전과소요 : 정원 대비 병과인원 부족 소요, 전과 희망자 간 상호 일치 시, \n신규 병과창설 등 인력 운영 필요시, 병과 전문자격 취득 및 교육 이수자 \n발생 시 등\n 신청방법\n•\x07각 군 전과 계획에 따라 변경 가능\n•병과전환(전과) 희망하는 병과에 대해 사전 충분한 이해 필요 \n각 군별 담당\n담당부서\n-③ 전과(병과 전환) 제도',
 '이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주요 항목은 **주둔지 단위 방문 교육**, **소집교육**, 그리고 **희망 인원**으로 구성되어 있

In [24]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다. 
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 현역병 복지에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법읋 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" 
# 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정
)





C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\3702810314.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\3702810314.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [25]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]Node 914537ad-0c1c-4379-81fb-6bebecae6bd8 does not have a summary. Skipping filtering.
Node 1e8542cb-e3a6-42ea-a11a-b73d8c598690 does not have a summary. Skipping filtering.
Node f14181c2-879c-4f40-bb79-8aa861905952 does not have a summary. Skipping filtering.
Node 2cc3d592-ff4b-4cf6-b45b-8f23d539898b does not have a summary. Skipping filtering.
Node d25ce8c2-cdbd-45b3-8620-faf012f97de2 does not have a summary. Skipping filtering.
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 1885.08it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generating Samples: 100%|██████████| 10/10 [00:03<00:00,  2.99it/s]


In [26]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='간부 복지혜택이 먼지 자세히 알수있나요? 그리고 간부 주거 지원제도도 포함해서 설명해주실수있나요?', retrieved_contexts=None, reference_contexts=['생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 지원제도\n-⑥ 맞춤형복지제도\n-⑦ 초급간부 재정안정지원\n-⑧ 군인공제회 회원복지부조\n-⑨ 군인공제회 건강검진 제휴 서비스\n-⑩ 군인공제회 콘도 및 제휴 복지 서비스\n-⑪ 장병e음 플랫폼 서비스\n02\nPart'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='간부 복지혜택에는 초급간부 보수 및 수당안내, 간부 복지혜택, 일-가정 양립 지원제도, 초급간부 대상 피복 및 급식 지원, 간부 주거 지원제도, 맞춤형복지제도, 초급간부 재정안정지원, 군인공제회 회원복지부조, 군인공제회 건강검진 제휴 서비스, 군인공제회 콘도 및 제휴 복지 서비스, 장병e음 플랫폼 서비스 등이 포함됩니다. 간부 주거 지원제도도 이 복지혜택 중 하나로 안내되고 있습니다.', rubrics=None, persona_name='Service Member Seeking Career Change', query_style='MISSPELLED', query_length='LONG'), synthesizer_name='single_hop_specific_query_synthesizer'), TestsetSample(eval_sample=SingleTurnSample(user_input='장병e음 플랫폼 서비스가 군 복무 중인 장병들에게 제공하는 주요 기능과 혜택은 무엇인가요?', retr

In [27]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 간부 복지혜택이 먼지 자세히 알수있나요? 그리고 간부 주거 지원제도도 포함해서 설명해주실수있나요?
Context: ['생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 지원제도\n-⑥ 맞춤형복지제도\n-⑦ 초급간부 재정안정지원\n-⑧ 군인공제회 회원복지부조\n-⑨ 군인공제회 건강검진 제휴 서비스\n-⑩ 군인공제회 콘도 및 제휴 복지 서비스\n-⑪ 장병e음 플랫폼 서비스\n02\nPart']
생성된 답변(정답): 간부 복지혜택에는 초급간부 보수 및 수당안내, 간부 복지혜택, 일-가정 양립 지원제도, 초급간부 대상 피복 및 급식 지원, 간부 주거 지원제도, 맞춤형복지제도, 초급간부 재정안정지원, 군인공제회 회원복지부조, 군인공제회 건강검진 제휴 서비스, 군인공제회 콘도 및 제휴 복지 서비스, 장병e음 플랫폼 서비스 등이 포함됩니다. 간부 주거 지원제도도 이 복지혜택 중 하나로 안내되고 있습니다.
평가대상 RAG의 답변: None
평가대상 RAG가 검색한 Context: None


In [28]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

In [29]:
eval_df.shape

(10, 7)

In [30]:
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,간부 복지혜택이 먼지 자세히 알수있나요? 그리고 간부 주거 지원제도도 포함해서 설명...,[생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-...,"간부 복지혜택에는 초급간부 보수 및 수당안내, 간부 복지혜택, 일-가정 양립 지원제...",Service Member Seeking Career Change,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,장병e음 플랫폼 서비스가 군 복무 중인 장병들에게 제공하는 주요 기능과 혜택은 무엇...,[생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-...,"장병e음 플랫폼 서비스는 생활편의 분야에서 제공되는 서비스 중 하나로, 군 복무 중...",Junior Military Officer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,개막식 뭐 하는 거야? 채용 설명회랑 무슨 관계 있어?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,"개막식은 행사 구성 중 하나로, 표에 따르면 개막식 아래에 채용 설명회가 포함되어 ...",Service Member Seeking Career Change,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,개막식에서 어떤 프로그램이 진행되나요?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,"개막식에서는 채용 설명회가 진행되며, 이는 채용 관련 정보를 제공하는 프로그램입니다.",Service Member Seeking Career Change,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,육군 병과 바꿀수 있나여?,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...,육군에서는 임관 이후에도 실무경험과 적성 등을 고려하여 병과 변경이 가능합니다.,Service Member Seeking Career Change,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
5,육군에서 병과를 변경하고 싶은 현역 군인이 지원 자격과 절차에 대해 자세히 설명해 ...,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...,"육군에서 병과를 변경하려면 원병과에서 1년 이상 근무한 경력이 있어야 하며, 희망 ...",Service Member Seeking Career Change,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
6,창업 인식 개셍 교욱 뭐에요?,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,"창업 인식 개선 교육은 주둔지 단위 방문 교육, 소집교육, 그리고 희망 인원으로 구...",Service Member Seeking Career Change,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
7,창업 인식 개셍 교욱이 뭐에요? 이거 신청하는 방법이나 누가 들을 수 있는지 궁금한...,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,"창업 인식 개선 교육은 표에 따르면 주둔지 단위 방문 교육, 소집교육, 그리고 희망...",Prospective Job Seeker,MISSPELLED,LONG,single_hop_specific_query_synthesizer
8,임신여군 보직조정 어떻게 되나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...",분만취약지 근무중인 임신여군의 경우 본인 희망시 분만가능 산부인과 인근지역(30분 ...,Service Member Seeking Career Change,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
9,군무원 임신 시 야간근무 제한이 있나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...",임신 중이거나 출산 후 1년이 지나지 않은 군무원의 경우 21시부터 08시까지 야간...,Prospective Job Seeker,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [31]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q)


In [32]:
# resp
resp['response']

'정보가 부족해 답을 할 수없습니다.'

In [33]:
# resp
resp['retrieved_context']

['02\nPart\n생\n활\n편\n의\n \n분\n야\n45 \n2026 초급간부 길라잡이\n-⑤ 간부 주거 지원제도\n 개 요 \n \x07 \x07군인이 안정된 주거생활을 함으로써 근무에 전념할 수 있도록 하기 위하여 \n주거시설을 제공하거나 민간주택 전·월세 지원\n 군 주거지원 종류\n구 분\n내 용\n군 주거시설\n공관, 지휘관관사, 일반관사, 간부숙소\n민간주택 전ㆍ월세\n (이자지원)\n협약은행에서 전ㆍ월세 자금을 지원하고 \n국가는 그 이자를 지원\n 지원대상\n구 분\n내 용\n군 주거\n시설\n관사\n부양가족이 있고 관리부대 근무지 내에\n자가주택을 보유하지 않은 군인\n간부\n숙소\n 미혼간부ㆍ가족과 별거하는 기혼간부\n민간주택 전ㆍ월세\n(이자지원)\n관리부대의 근무지 내에 관사가 부족하여 \n민간주택 전ㆍ월세 지원이 필요한 경우\n 신청방법 \n • “국방인사정보체계”를 통해 신청 \n 초급간부 주거여건 개선 지속 추진\n •전용면적 확대(미혼 간부숙소 18㎡→ 24㎡) \n •비품개선(실별 세탁기, 취사용 인덕션, 빌트인가구 설치)\n국방부 군 주거정책과 주거관리담당 | 900-5472\n담당부서',
 '이 표는 **「생활편의 분야」**에 포함된 군 간부·초급간부 대상 지원 항목을 정리한 목차성 표로, **보수·수당 안내, 복지혜택, 일·가정 양립 지원제도, 피복 및 급식 지원, 주거 지원, 맞춤형복지제도, 재정안정지원** 등이 핵심 내용입니다. 또한 **군인공제회 회원복지부조, 건강검진 제휴 서비스, 콘도 및 제휴 복지 서비스, 장병e음 플랫폼 서비스**까지 포함해 총 **11개 세부 항목(\ue425-①~\ue425-⑪)**으로 구성되어 있습니다. 컬럼은 별도의 수치 데이터를 담기보다, 이 분야의 **세부 주제 목록과 구성 범위**를 보여주는 역할을 합니다.',
 '생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 지원제

In [36]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp["response"])
    retrieved_context_list.append(resp["retrieved_context"])

In [35]:
print(len(response_list), len(retrieved_context_list))

10 10


In [37]:
response_list

['정보가 부족해 답을 할 수없습니다.',
 '장병e음 플랫폼 서비스는 군 복무 중인 장병들에게 국방 관련 서비스를 인터넷·모바일 환경에서 통합 제공하는 플랫폼입니다.\n\n주요 기능과 혜택은 다음과 같습니다.\n\n- **인사**: 신분 인증 및 확인(신분증), 통합 회원 관리(통합 로그인), 급여명세서 조회\n- **소통**: 알림톡 발송, 각 군 훈련소 연결, 소통과 공감앱 연결, 헬프콜 연결\n- **행정**: 휴가/외박/출장 조회, 각군 모집안내 연결, 민방위훈련 유예(면제) 신청 연결\n- **복지**: 자녀 기숙사 신청, 군 인터넷 쇼핑몰(와몰) 연결, 체력단련장 예약, 호텔·콘도 예약, 웨딩/연회 예약, 항공 예약, 해상 예약, 버스/열차 예매, 전세객차 예매, 영화할인 신청, 군인공제회 예금조회, 맞춤형 e북 지원, 내일준비적금 가입\n- **교육**: 국가기술자격검정 연결, 제대군인 취업센터 연결, 일반강좌 신청, 대학 원격강좌 신청, 진로설계유형 검사, 디지털 독서지원 신청\n- **의료**: 군 병원 예약, 민간병원 진료비 청구, 심리 검사\n\n즉, 복무 중 장병들은 **급여·휴가·복지·교육·의료 관련 서비스를 한곳에서 편리하게 이용할 수 있는 것**이 주요 혜택입니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '개막식에서 진행되는 프로그램은 **채용 설명회**입니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '육군의 병과 변경은 **전과(병과 전환) 제도**에 해당합니다.  \n주어진 자료 기준으로 보면, **육군 장교**의 지원자격은 다음과 같습니다.\n\n### 지원 자격\n- **원 병과에서 1년 이상 근무한 경우**\n- **희망 병과 관련 자격을 취득하고 교육을 이수한 자 등**\n\n### 전과 소요가 생기는 경우\n다음과 같은 경우 전과가 가능합니다.\n- **정원 대비 병과 인원이 부족한 경우**\n- **전과 희망자 간 상호 일치가 있는 경우**\n- **신규 병과 창설 등 인력 운영상 필요가 있는 경우*

In [38]:
retrieved_context_list

[['02\nPart\n생\n활\n편\n의\n \n분\n야\n45 \n2026 초급간부 길라잡이\n-⑤ 간부 주거 지원제도\n 개 요 \n \x07 \x07군인이 안정된 주거생활을 함으로써 근무에 전념할 수 있도록 하기 위하여 \n주거시설을 제공하거나 민간주택 전·월세 지원\n 군 주거지원 종류\n구 분\n내 용\n군 주거시설\n공관, 지휘관관사, 일반관사, 간부숙소\n민간주택 전ㆍ월세\n (이자지원)\n협약은행에서 전ㆍ월세 자금을 지원하고 \n국가는 그 이자를 지원\n 지원대상\n구 분\n내 용\n군 주거\n시설\n관사\n부양가족이 있고 관리부대 근무지 내에\n자가주택을 보유하지 않은 군인\n간부\n숙소\n 미혼간부ㆍ가족과 별거하는 기혼간부\n민간주택 전ㆍ월세\n(이자지원)\n관리부대의 근무지 내에 관사가 부족하여 \n민간주택 전ㆍ월세 지원이 필요한 경우\n 신청방법 \n • “국방인사정보체계”를 통해 신청 \n 초급간부 주거여건 개선 지속 추진\n •전용면적 확대(미혼 간부숙소 18㎡→ 24㎡) \n •비품개선(실별 세탁기, 취사용 인덕션, 빌트인가구 설치)\n국방부 군 주거정책과 주거관리담당 | 900-5472\n담당부서',
  '이 표는 **「생활편의 분야」**에 포함된 군 간부·초급간부 대상 지원 항목을 정리한 목차성 표로, **보수·수당 안내, 복지혜택, 일·가정 양립 지원제도, 피복 및 급식 지원, 주거 지원, 맞춤형복지제도, 재정안정지원** 등이 핵심 내용입니다. 또한 **군인공제회 회원복지부조, 건강검진 제휴 서비스, 콘도 및 제휴 복지 서비스, 장병e음 플랫폼 서비스**까지 포함해 총 **11개 세부 항목(\ue425-①~\ue425-⑪)**으로 구성되어 있습니다. 컬럼은 별도의 수치 데이터를 담기보다, 이 분야의 **세부 주제 목록과 구성 범위**를 보여주는 역할을 합니다.',
  '생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-가정 양립 지원제도\n-④ 초급간부 대상 피복 및 급식 지원\n-⑤ 간부 주거 

In [39]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,간부 복지혜택이 먼지 자세히 알수있나요? 그리고 간부 주거 지원제도도 포함해서 설명...,[생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-...,"간부 복지혜택에는 초급간부 보수 및 수당안내, 간부 복지혜택, 일-가정 양립 지원제...",Service Member Seeking Career Change,MISSPELLED,LONG,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[02\nPart\n생\n활\n편\n의\n \n분\n야\n45 \n2026 초급간부...
1,장병e음 플랫폼 서비스가 군 복무 중인 장병들에게 제공하는 주요 기능과 혜택은 무엇...,[생활편의 분야\n-① 초급간부 보수 및 수당안내\n-② 간부 복지혜택\n-③ 일-...,"장병e음 플랫폼 서비스는 생활편의 분야에서 제공되는 서비스 중 하나로, 군 복무 중...",Junior Military Officer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer,장병e음 플랫폼 서비스는 군 복무 중인 장병들에게 국방 관련 서비스를 인터넷·모바일...,[2026 병 복지 길라잡이 27\n장병e음 플랫폼 서비스\n개 요\n 장병들에게 ...
2,개막식 뭐 하는 거야? 채용 설명회랑 무슨 관계 있어?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,"개막식은 행사 구성 중 하나로, 표에 따르면 개막식 아래에 채용 설명회가 포함되어 ...",Service Member Seeking Career Change,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...
3,개막식에서 어떤 프로그램이 진행되나요?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,"개막식에서는 채용 설명회가 진행되며, 이는 채용 관련 정보를 제공하는 프로그램입니다.",Service Member Seeking Career Change,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,개막식에서 진행되는 프로그램은 **채용 설명회**입니다.,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...
4,육군 병과 바꿀수 있나여?,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...,육군에서는 임관 이후에도 실무경험과 적성 등을 고려하여 병과 변경이 가능합니다.,Service Member Seeking Career Change,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[Part 03 인사제도/근무 분야\n46 2026 병 복지 길라잡이\n병역감면원서...
5,육군에서 병과를 변경하고 싶은 현역 군인이 지원 자격과 절차에 대해 자세히 설명해 ...,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...,"육군에서 병과를 변경하려면 원병과에서 1년 이상 근무한 경력이 있어야 하며, 희망 ...",Service Member Seeking Career Change,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer,육군의 병과 변경은 **전과(병과 전환) 제도**에 해당합니다. \n주어진 자료 ...,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...
6,창업 인식 개셍 교욱 뭐에요?,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,"창업 인식 개선 교육은 주둔지 단위 방문 교육, 소집교육, 그리고 희망 인원으로 구...",Service Member Seeking Career Change,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,"창업 인식 개선 교육은 장병들에게 **창업의 이해, 창업시장 동향, 창업준비 내용,...",[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...
7,창업 인식 개셍 교욱이 뭐에요? 이거 신청하는 방법이나 누가 들을 수 있는지 궁금한...,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,"창업 인식 개선 교육은 표에 따르면 주둔지 단위 방문 교육, 소집교육, 그리고 희망...",Prospective Job Seeker,MISSPELLED,LONG,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...
8,임신여군 보직조정 어떻게 되나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...",분만취약지 근무중인 임신여군의 경우 본인 희망시 분만가능 산부인과 인근지역(30분 ...,Service Member Seeking Career Change,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,임신여군은 **희망 시** 분만가능 산부인과 **인근지역(30분 이내)**으로 **...,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n..."
9,군무원 임신 시 야간근무 제한이 있나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...",임신 중이거나 출산 후 1년이 지나지 않은 군무원의 경우 21시부터 08시까지 야간...,Prospective Job Seeker,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,"네, 있습니다. \n지휘관은 **임신 중이거나 출산 후 1년이 지나지 않은 군인·...","[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n..."


In [41]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

In [42]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\3378832119.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\3378832119.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\3378832119.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\P

In [43]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_9776\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
Evaluating:  72%|███████▎  | 29/40 [02:46<00:57,  5.22s/it]Exception raised in Job[6]: TimeoutError()
Exception raised in Job[9]: TimeoutErro

In [44]:
eval_result

{'context_recall': 0.8889, 'llm_context_precision_with_reference': 0.7618, 'faithfulness': 0.6667, 'answer_relevancy': 0.3628}

In [45]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,간부 복지혜택이 먼지 자세히 알수있나요? 그리고 간부 주거 지원제도도 포함해서 설명...,[02\nPart\n생\n활\n편\n의\n \n분\n야\n45 \n2026 초급간부...,정보가 부족해 답을 할 수없습니다.,"간부 복지혜택에는 초급간부 보수 및 수당안내, 간부 복지혜택, 일-가정 양립 지원제...",1.0,1.000000,0.0,0.000000
1,장병e음 플랫폼 서비스가 군 복무 중인 장병들에게 제공하는 주요 기능과 혜택은 무엇...,[2026 병 복지 길라잡이 27\n장병e음 플랫폼 서비스\n개 요\n 장병들에게 ...,장병e음 플랫폼 서비스는 군 복무 중인 장병들에게 국방 관련 서비스를 인터넷·모바일...,"장병e음 플랫폼 서비스는 생활편의 분야에서 제공되는 서비스 중 하나로, 군 복무 중...",1.0,NaN,NaN,0.786760
2,개막식 뭐 하는 거야? 채용 설명회랑 무슨 관계 있어?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,정보가 부족해 답을 할 수없습니다.,"개막식은 행사 구성 중 하나로, 표에 따르면 개막식 아래에 채용 설명회가 포함되어 ...",1.0,NaN,0.0,0.000000
3,개막식에서 어떤 프로그램이 진행되나요?,[이 표는 문서의 행사 구성 중 **개막식** 항목 아래에 포함된 세부 프로그램을 ...,개막식에서 진행되는 프로그램은 **채용 설명회**입니다.,"개막식에서는 채용 설명회가 진행되며, 이는 채용 관련 정보를 제공하는 프로그램입니다.",1.0,0.700000,1.0,0.930729
4,육군 병과 바꿀수 있나여?,[Part 03 인사제도/근무 분야\n46 2026 병 복지 길라잡이\n병역감면원서...,정보가 부족해 답을 할 수없습니다.,육군에서는 임관 이후에도 실무경험과 적성 등을 고려하여 병과 변경이 가능합니다.,0.0,0.000000,NaN,0.000000
5,육군에서 병과를 변경하고 싶은 현역 군인이 지원 자격과 절차에 대해 자세히 설명해 ...,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n15 \n...,육군의 병과 변경은 **전과(병과 전환) 제도**에 해당합니다. \n주어진 자료 ...,"육군에서 병과를 변경하려면 원병과에서 1년 이상 근무한 경력이 있어야 하며, 희망 ...",1.0,1.000000,NaN,0.000000
6,창업 인식 개셍 교욱 뭐에요?,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,"창업 인식 개선 교육은 장병들에게 **창업의 이해, 창업시장 동향, 창업준비 내용,...","창업 인식 개선 교육은 주둔지 단위 방문 교육, 소집교육, 그리고 희망 인원으로 구...",NaN,0.833333,1.0,0.567798
7,창업 인식 개셍 교욱이 뭐에요? 이거 신청하는 방법이나 누가 들을 수 있는지 궁금한...,[이 표는 **창업 인식 개선 교육**의 운영 방식을 간단히 정리한 내용입니다. 주...,정보가 부족해 답을 할 수없습니다.,"창업 인식 개선 교육은 표에 따르면 주둔지 단위 방문 교육, 소집교육, 그리고 희망...",1.0,0.755556,NaN,0.000000
8,임신여군 보직조정 어떻게 되나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...",임신여군은 **희망 시** 분만가능 산부인과 **인근지역(30분 이내)**으로 **...,분만취약지 근무중인 임신여군의 경우 본인 희망시 분만가능 산부인과 인근지역(30분 ...,1.0,1.000000,1.0,0.601752
9,군무원 임신 시 야간근무 제한이 있나요?,"[가족수당\n•첫째 월 5만원, 둘째 월 8만원, 셋째 이상 1명당 월 12만원\n...","네, 있습니다. \n지휘관은 **임신 중이거나 출산 후 1년이 지나지 않은 군인·...",임신 중이거나 출산 후 1년이 지나지 않은 군무원의 경우 21시부터 08시까지 야간...,1.0,0.805556,1.0,0.741394
